In [ ]:
import math
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import ssl
import os
from pathlib import Path
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
import pandas as pd
from datetime import datetime
import pickle
import re
import pyxdameraulevenshtein
# import apsw  # Commented out - not needed for CSV processing
import sys
import numpy as np
import corp_simplify_utils
import seaborn as sns
import matplotlib.pyplot as plt
import pyreadr
from collections import Counter

# nlp
import spacy
from spacy import displacy
# Remove problematic direct import: import en_core_web_lg
# Use spacy.load() instead (loaded below when needed)

# analysis/regressions
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind

# from statsmodels.graphics.gofplots import qqplot_2samples
from scipy import stats
from joypy import joyplot
from matplotlib import cm

from datetime import date
today_for_filenames = date.today()
curr_date = str(today_for_filenames.strftime("%Y%m%d"))


NUMBER_OF_MATCHES_TO_RECORD = 10
punc_remove_re = re.compile(r'\W+')
corp_re = re.compile('( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc))+$')
and_re = re.compile(' & ')
punc1_re = re.compile(r"(?<=\S)['\u00B4\.](?=\S)")  # Fixed unicode character
punc2_re = re.compile(r"[\s\.,:;/'\"`\u00B4\u2018\u2019\u201C\u201D\(\)\[\]\{\}_\u2014\-?$=!]+")  # Fixed unicode characters

STOPWORDS = nltk.corpus.stopwords.words('english')
STOPWORDS.remove("am")
STOPWORDS.remove("up")
STOPWORDS.remove("in")
STOPWORDS.remove("on")
STOPWORDS.remove("all")
STOPWORDS.remove("any")
STOPWORDS.remove("most")
STOPWORDS.remove("no")
STOPWORDS.remove("nor")
STOPWORDS.remove("own")
STOPWORDS.remove("same")
STOPWORDS.remove("so")
STOPWORDS.remove("very")
STOPWORDS.remove("s")
STOPWORDS.remove("t")
STOPWORDS.remove("d")
STOPWORDS.remove("ll")
STOPWORDS.remove("m")
STOPWORDS.remove("o")
STOPWORDS.remove("re")
STOPWORDS.remove("ve")
STOPWORDS.remove("y")

#compile regex patterns to reuse
STOPWORD_RE = re.compile(r'\b(the|of|and|in|on)\b', re.IGNORECASE)
CORP_SUFFIX_RE = re.compile(r'\b(inc|corp|ltd|llc|plc|co|company|limited)\b', re.IGNORECASE)
PDF_PATTERN_RE = re.compile(r'\s[0-9]*\s[km]b\s*pdf', re.IGNORECASE)
PUNCT_RE = re.compile(r'[^\w\s-]')  # match punctuation
MULTISPACE_RE = re.compile(r'\s+')

stopword_re_str = r""
for word in STOPWORDS:
	stopword_re_str += r'\b' + word + r'\b|'
stopword_re = re.compile(stopword_re_str[:-1]) # The negative 1 is for the fencepost |

# Commented out - these paths don't exist in your project
# BASE_DIR = "/Users/aawesomez/Documents/UROP/NLP-regextable/"
# DB_PATH = BASE_DIR + "Data/master.sqlite"
# LAST_SAVE_DATASET_DATE = "20220402"

# Function to calculate longest common substring, from https://www.geeksforgeeks.org/print-longest-common-substring/
# function to find and print 
# the longest common substring of
# X[0..m-1] and Y[0..n-1]
def get_longest_common_substring(X, Y, m, n):
 
    # Create a table to store lengths of
    # longest common suffixes of substrings.
    # Note that LCSuff[i][j] contains length
    # of longest common suffix of X[0..i-1] and
    # Y[0..j-1]. The first row and first
    # column entries have no logical meaning,
    # they are used only for simplicity of program
    LCSuff = [[0 for i in range(n + 1)]
                 for j in range(m + 1)]
 
    # To store length of the
    # longest common substring
    length = 0
 
    # To store the index of the cell
    # which contains the maximum value.
    # This cell's index helps in building
    # up the longest common substring
    # from right to left.
    row, col = 0, 0
 
    # Following steps build LCSuff[m+1][n+1]
    # in bottom up fashion.
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif X[i - 1] == Y[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                if length < LCSuff[i][j]:
                    length = LCSuff[i][j]
                    row = i
                    col = j
            else:
                LCSuff[i][j] = 0
 
    # if true, then no common substring exists
    if length == 0:
        return ""
 
    # allocate space for the longest
    # common substring
    resultStr = ['0'] * length
 
    # traverse up diagonally form the
    # (row, col) cell until LCSuff[row][col] != 0
    while LCSuff[row][col] != 0:
        length -= 1
        resultStr[length] = X[row - 1] # or Y[col-1]
 
        # move diagonally up to previous cell
        row -= 1
        col -= 1
 
    # required longest common substring
    longest_common_substring = ''.join(resultStr)

    return longest_common_substring


# Function from Brad Hackinen's NAMA
def basicHash(s):
    '''
    A simple case and puctuation-insensitive hash
    '''
    s = s.lower()
    s = re.sub(and_re,' and ',s)
    s = re.sub(punc1_re,'',s)
    s = re.sub(punc2_re,' ',s)
    s = s.strip()

    return s

# Function from Brad Hackinen's NAMA
def corpHash(s):
    '''
    A hash function for corporate subsidiaries
    Insensitive to
        -case & punctation
        -'the' prefix
        -common corporation suffixes, including 'holding co'
    '''
    s = basicHash(s)
    if s.startswith('the '):
        s = s[4:]

    s = re.sub(corp_re,'',s,count=1)

    return s

# function to clean org names
def clean_fin_org_names(name: str) -> str:
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    
    # James strip metadata from name
    # name = name.split(',')[0]
    #Remove patterns like "10 kb pdf"
    name = PDF_PATTERN_RE.sub("", name)

    #Unicode and punctuation cleanup
    name = corp_simplify_utils.normalize_unicode(name)
    name = PUNCT_RE.sub(" ", name)

    #Remove corporate suffixes and stopwords
    name = CORP_SUFFIX_RE.sub("", name)
    name = STOPWORD_RE.sub("", name)

    #Normalize spacing and lowercase
    name = MULTISPACE_RE.sub(" ", name).strip().lower()

    return name


In [ ]:
# Locate data directory and read in data files

current_dir = Path(os.getcwd()).parent
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()
    

In [ ]:
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(clean_fin_org_names)
fdic_df['std_name'] = fdic_df['NAME'].apply(clean_fin_org_names)
sec_df['std_name'] = sec_df['Name'].apply(clean_fin_org_names)
cik_df['std_name'] = cik_df['company_name'].apply(clean_fin_org_names)

In [ ]:
cik_df.head(10)

In [ ]:
sec_df.head(10)

In [ ]:
compustat_df.head(10)

In [ ]:
fdic_df.head(10)

In [ ]:
# Test if CIK is already in SEC
sec_ciks = set(sec_df['CIK'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"CIKs in sec_df: {len(sec_ciks)}")
print(f"CIKs in cik_df: {len(cik_ciks)}")
print(f"Is SEC_Institutions.csv a subset of CIK.csv? {sec_ciks.issubset(cik_ciks)}")

compustat_ciks = set(compustat_df['cik'].dropna())
print(f"CIKs in compustat_df: {len(compustat_ciks)}")
print(f"Is CompustatNames.csv a subset of CIK.csv? {compustat_ciks.issubset(cik_ciks)}")


SEC_Institutions.csv is a subset of CIK.csv --- > Don't need to merge SEC into the crosswalk. 

In [ ]:
# renaming columns for consistency
compustat_temp = compustat_df[['std_name', 'conm', 'tic', 'cusip', 'cik']].rename(columns={'conm': 'raw_name'})
compustat_temp['source'] = 'compustat'

fdic_temp = fdic_df[['std_name', 'NAME']].rename(columns={'NAME': 'raw_name'})
fdic_temp['source'] = 'fdic'

cik_temp = cik_df[['std_name', 'company_name', 'cik']].rename(columns={'company_name': 'raw_name'})
cik_temp['source'] = 'cik'

# Combine all into one long dataframe
all_names_df = pd.concat([compustat_temp, fdic_temp, cik_temp], ignore_index=True)

# Drop any rows where cleaning failed (no std_name)
all_names_df = all_names_df.dropna(subset=['std_name'])
# Remove any empty std_name entries
all_names_df = all_names_df[all_names_df['std_name'] != ""]

# Cleaning up CIKs and FED_RSSD to be strings without decimal points
for col in ['CIK', 'FED_RSSD']:
    if col in all_names_df.columns:
        # Convert to string after converting to int to remove any decimal points
        all_names_df[col] = all_names_df[col].dropna().astype(float).astype(int).astype(str)

print(f"Total entries to match: {len(all_names_df)}")
all_names_df.head(20)

In [ ]:
grouped_by_cik_id = all_names_df.groupby('cik')
confident_matches = []

for cik_value, group in grouped_by_cik_id:
    if len(group) > 1:
        # Aggregate the data based on cik
        keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique())
        }
        confident_matches.append(keys)
        
pd.set_option('display.max_colwidth', None)
confident_crosswalk_id_based = pd.DataFrame(confident_matches)
print(f"Found {len(confident_crosswalk_id_based)} confident entity clusters.")

In [ ]:
confident_crosswalk_id_based.head(120)

Able to create a table of 56760 companies based on CIK id.

In [ ]:
# Find all the rows where there is duplicate CIK in one of the dataframes
# In other words, there are multiple aliases, but coming from the same source

confident_crosswalk_id_based[
    (confident_crosswalk_id_based['aliases'].str.contains('\|')) & 
    (~confident_crosswalk_id_based['sources'].str.contains(r','))
].head(20)

There appears to be many aliases for the same CIK ID in the cik.csv file

In [ ]:
### TO DO: fix the formating of the merged table

# # Create a copy of the confident_crosswalk_id_based DataFrame
crosswalk_with_fdic = confident_crosswalk_id_based.copy()

# Group the FDIC data by the standardized name
fdic_grouped = fdic_df.groupby('std_name')['NAME'].apply(list).reset_index()
fdic_grouped = fdic_grouped.rename(columns={'NAME': 'all_fdic_aliases'})

crosswalk_with_fdic = crosswalk_with_fdic.merge(
    fdic_grouped,
    how='left',
    left_on='standardized_names',
    right_on='std_name'
)

not_na_filter = crosswalk_with_fdic['all_fdic_aliases'].notna()

aliases_series = crosswalk_with_fdic.loc[not_na_filter, 'aliases'].fillna('').astype(str)
fdic_aliases_series = crosswalk_with_fdic.loc[not_na_filter, 'all_fdic_aliases'].astype(str)
crosswalk_with_fdic.loc[not_na_filter, 'aliases'] = \
    aliases_series + ', ' + fdic_aliases_series
    
crosswalk_with_fdic.loc[not_na_filter, 'sources'] = \
    crosswalk_with_fdic['sources'].fillna('') + ',FDIC'

    
crosswalk_with_fdic

In [ ]:
crosswalk_with_fdic[crosswalk_with_fdic['all_fdic_aliases'].notna()]

In [ ]:
# clean up the crosswalk of FDIC + CIK + Compustat 

In [ ]:
# need to isolate remaining data that couldn't be matched by CIK
# checks the size of each group by CIK
cik_group_sizes = all_names_df.groupby('cik')['cik'].transform('size')
processed_rows_mask = (cik_group_sizes > 1)
remaining_df = all_names_df[~processed_rows_mask].copy()

total_rows = len(all_names_df)
processed_rows_count = processed_rows_mask.sum()
remaining_rows_count = len(remaining_df)
print(f"Total rows: {total_rows}")
print(f"Processed rows (matched by CIK): {processed_rows_count}")
print(f"Remaining rows to process: {remaining_rows_count}")

In [ ]:
# Remaining rows to process will go through other matching methods including fuzzy matching and regex-based matching

grouped_by_std_name = remaining_df.groupby('std_name')
confident_matches_std_name = []

for name, group in grouped_by_std_name:
    # A "match" means this std_name appeared in more than one row
    if len(group) > 1:
        # Aggregate all unique keys and aliases
        keys = {
            'std_name': name,
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique()),
            'CIK': group['cik'].dropna().unique().tolist(),
        }
        confident_matches_std_name.append(keys)
pd.set_option('display.max_rows', None)
confident_crosswalk_std_name = pd.DataFrame(confident_matches_std_name)
print(f"  Found {len(confident_crosswalk_std_name)} additional clusters based on std_name.")
print(confident_crosswalk_std_name.head(100))
confident_crosswalk_std_name

In [ ]:
cik_df[cik_df['company_name'] == 'ADAMS JOHN S']

In [ ]:
# from collections import Counter
# import pandas as pd
# from tqdm.auto import tqdm

# def get_match_candidate_score(frequency_dict, org_name, candidate_match_name):
#     if not isinstance(org_name, str):
#         org_name = ""
#     if not isinstance(candidate_match_name, str):
#         candidate_match_name = ""
    
#     if not org_name or not candidate_match_name:
#         return 0.0
    
#     org_tokens = org_name.split(' ')
    
#     # tokenize the candidate match
#     candidate_match_tokens = set(candidate_match_name.split(" "))

#     # Calculate the match score
#     total_inverse_frequency = 0
#     total_matching_inverse_frequency = 0
#     tokenized_name = org_tokens
#     for token in tokenized_name:
#         token_frequency = frequency_dict.get(token, 999999) # if token not found, give high frequency to ignore it
#         total_inverse_frequency += 1.0/token_frequency
#         if token in candidate_match_tokens:
#             total_matching_inverse_frequency += 1.0/token_frequency
#     match_score = total_matching_inverse_frequency / total_inverse_frequency

#     # added by James
#     weight = 1/len(org_name)
#     longest_common_substring = get_longest_common_substring(org_name, candidate_match_name, len(org_name), len(candidate_match_name))
#     match_score -= weight * len(candidate_match_name)/len(longest_common_substring) - weight

#     return match_score

# still_unmatched_df = remaining_df[~remaining_df['std_name'].isin(confident_crosswalk_std_name['std_name'])].copy().sample(frac=0.01, random_state=42)  # Sample 5% for testing
# print(f"Records remaining after std_name matching: {len(still_unmatched_df)}")
# all_tokens = []
# for name in still_unmatched_df['std_name']:
#     if isinstance(name, str) and name:
#         all_tokens.extend(name.lower().split(' '))

# token_frequency_dict = Counter(all_tokens)

# # --- 2. Create Candidate Match Dictionary (Reverse Index) ---
# candidate_match_dict = {}

# # Iterate over all unmatched records to build the index
# for idx, row in still_unmatched_df.iterrows():
#     # Store necessary data: (unique_id, std_name, original_raw_name, source)
#     candidate_tuple = (
#         f"{row['source']}-{idx}", 
#         row['std_name'].lower(), 
#         row['raw_name'], 
#         row['source']
#     )
    
#     for token in row['std_name'].lower().split(" "):
#         if token not in candidate_match_dict:
#             candidate_match_dict[token] = []
        
#         # Add candidate to the list for that token
#         if candidate_tuple not in candidate_match_dict[token]:
#              candidate_match_dict[token].append(candidate_tuple)


# # Set your desired threshold for a match
# MATCH_THRESHOLD = 0.95 
# high_confidence_matches = []

# # Loop through every company name in the DataFrame
# for idx, row in tqdm(still_unmatched_df.iterrows(), total=len(still_unmatched_df), desc="Entity Matching"):
#     org_name = row['std_name'].lower()
    
#     # --- A. Find candidates using top 2 rarest tokens (Optimization) ---
#     org_tokens = org_name.split(" ")
#     org_token_frequencies = sorted(
#         [(token, token_frequency_dict.get(token, 1)) for token in org_tokens],
#         key=lambda x: x[1] # Sort by frequency (rarest first)
#     )
    
#     candidate_set = set() 
    
#     # Use the two most unique tokens to fetch potential candidates from the index
#     for most_unique_token, _ in org_token_frequencies[:2]:
#         if most_unique_token in candidate_match_dict:
#             for candidate_tuple in candidate_match_dict[most_unique_token]:
#                 candidate_idx = candidate_tuple[0].split('-')[-1]
#                 # Exclude self-comparison
#                 if str(idx) != candidate_idx: 
#                     candidate_set.add(candidate_tuple)
    
#     # --- B. Score Candidates and Store Matches ---
#     for candidate_tuple in candidate_set:
#         unique_id, candidate_name, original_name_candidate, source_candidate = candidate_tuple
        
#         # Call the single-score function to get similarity
#         match_score = get_match_candidate_score(
#             token_frequency_dict, org_name, candidate_name)
        
#         # Filter for high-confidence matches
#         if match_score >= MATCH_THRESHOLD:
#             # Record the full details of the high-confidence match
#             high_confidence_matches.append({
#                 'Name_1_STD': org_name,
#                 'Name_1_RAW': row['raw_name'],
#                 'Name_1_Source': row['source'],
#                 'Name_2_STD': candidate_name,
#                 'Name_2_RAW': original_name_candidate,
#                 'Name_2_Source': source_candidate,
#                 'Similarity_Score': match_score,
#             })

# # --- 3. Final DataFrame ---
# match_candidates_df = pd.DataFrame(high_confidence_matches)
# print("Entity Matching Complete. High-Confidence Matches DataFrame:")
# print(match_candidates_df.head(20))




